In [1]:
# Installing required libraries
!pip install -q transformers datasets accelerate evaluate torch

In [2]:
!pip uninstall -y transformers
!pip install -q transformers

Found existing installation: transformers 4.57.1
Uninstalling transformers-4.57.1:
  Successfully uninstalled transformers-4.57.1


In [3]:
# Loading required libraries
import torch
from transformers import BertTokenizerFast, BertForSequenceClassification, TrainingArguments, Trainer
from datasets import load_dataset
import numpy as np

In [4]:
# Loading IMDb dataset from Hugging Face Datasets
# Dataset link: https://huggingface.co/datasets/imdb
dataset = load_dataset('imdb')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [5]:
# Loading BERT tokenizer and model from Hugging Face
# Model link: https://huggingface.co/bert-base-uncased
model_name = "bert-base-uncased"
tokenizer = BertTokenizerFast.from_pretrained(model_name)
model = BertForSequenceClassification.from_pretrained(model_name, num_labels=2)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [6]:
# Tokenizing text data for BERT input
max_length = 128

def tokenize_batch(example):
    return tokenizer(
        example["text"],
        truncation=True,                 # trim long reviews
        padding="max_length",            # pad short reviews
        max_length=max_length
    )

# Apply tokenizer to the whole dataset
tokenized = dataset.map(tokenize_batch, batched=True)
tokenized = tokenized.remove_columns(["text"])  # keep only ids, mask, label
tokenized.set_format("torch")

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [7]:
# Preparing training and evaluation datasets
train_dataset = tokenized["train"]
eval_dataset = tokenized["test"]

In [11]:
# Defining training configuration
training_args = TrainingArguments(
    output_dir="./results",              # directory to save checkpoints
    eval_strategy="epoch",         # evaluate after each epoch
    per_device_train_batch_size=8,       # training batch size
    per_device_eval_batch_size=16,       # evaluation batch size
    num_train_epochs=2,                  # total number of epochs
    learning_rate=2e-5,                  # learning rate
    save_total_limit=1,                  # keep last checkpoint only
    logging_steps=100,                   # log every 100 steps
)

In [13]:
!pip install wandb

In [ ]:
# Creating Trainer object and starting training
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer
)

trainer.train()

/tmp/ipython-input-81393364.py:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ewitcselabbatch2 (ewitcselabbatch2-student) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss
